# In-Context Learning: Prompt Construction
Prerequisites: prompt examples and dictionaries. Estimated time: 30 minutes. Objectives: format demonstrations into a prompt, enforce a context budget, and evaluate a deterministic mock parser without a model or API.

## Concept recap and assumptions
In-context learning conditions a model on instructions and demonstrations provided at inference time; the model weights are unchanged. This notebook studies prompt construction and example ordering with a deterministic mock that matches an input pattern. It cannot demonstrate learned behavior or predict a language model's response.

In [ ]:
def build_prompt(examples, query, max_examples):
    if max_examples < 0:
        raise ValueError('max_examples must be non-negative')
    selected = examples[:max_examples]
    lines = ['Classify the message as positive or negative.']
    lines += [f'Message: {text}\nLabel: {label}' for text, label in selected]
    lines += [f'Message: {query}', 'Label:']
    return '\n'.join(lines)

def mock_completion(prompt):
    # Mock recognizes only an exact demonstrated message; no model inference occurs.
    for line, next_line in zip(prompt.splitlines(), prompt.splitlines()[1:]):
        if line.startswith('Message: ') and next_line.startswith('Label: ') and next_line != 'Label:':
            continue
    query = prompt.split('Message: ')[-1].split('\nLabel:')[0]
    for line, label in reversed([(line[9:], nxt[7:]) for line, nxt in zip(prompt.splitlines(), prompt.splitlines()[1:]) if line.startswith('Message: ') and nxt.startswith('Label: ') and nxt != 'Label:']):
        if line == query:
            return label
    return 'unknown'

examples = [('great service', 'positive'), ('late parcel', 'negative')]
prompt = build_prompt(examples, 'great service', max_examples=2)
print(prompt)

## Baseline assertions
The useful artifact here is the constructed context: instruction, demonstrations, and a query with an answer slot. The mock only echoes a label for an exact query already present in demonstrations; that behavior is a test fixture, not in-context learning.

In [ ]:
assert prompt.startswith('Classify the message')
assert prompt.endswith('Message: great service\nLabel:')
assert mock_completion(prompt) == 'positive'
assert mock_completion(build_prompt(examples, 'novel message', 2)) == 'unknown'

## Comparison and metric
Evaluate prompt design on fixed cases using exact-match accuracy and prompt length as a simple budget proxy. The score below measures only the mock's lookup behavior. Real models need repeated trials where sampling is enabled and should be evaluated against a held-out set with order and wording variants.

In [ ]:
cases = [('great service', 'positive'), ('late parcel', 'negative'), ('new phrase', 'unknown')]
results = [mock_completion(build_prompt(examples, q, 2)) for q, _ in cases]
mock_accuracy = sum(a == y for a, (_, y) in zip(results, cases)) / len(cases)
prompt_chars = len(prompt)
assert results == ['positive', 'negative', 'unknown']
assert mock_accuracy == 1.0 and prompt_chars > 0
# Failure cases: empty context still has an answer slot; negative budgets are invalid.
assert mock_completion(build_prompt([], 'unseen', 0)) == 'unknown'
try:
    build_prompt(examples, 'x', -1)
except ValueError:
    pass
else:
    raise AssertionError('negative example budget should fail')

## Exercises
1. Reverse demonstration order and confirm the prompt changes while the mock's exact-match output does not. 2. Add a character budget and drop whole examples without truncating labels. 3. Create conflicting duplicate demonstrations and define an explicit resolution policy. Expected properties: the query remains last, the budget is respected, and the evaluation score is clearly labeled as mock-only.

## Summary, next lesson, and practice
In-context learning uses examples placed in context at inference time; prompt selection, order, and budget matter. Our deterministic mock validates formatting and control flow only. Next: Chain-of-Thought methods and prompt optimization. Practice: [In-context learning interview drill](../interview-prep/curriculum-practice.md#in-context-learning).